In [15]:
import pandas as pd

# Load the dataset (upload fake_ae_dataset.xlsx to Colab first)
df = pd.read_excel('fake_ae_dataset.xlsx')

# Add a unique patient ID so reshaping has a unique key for each row
df.insert(0, 'Patient', ['P' + str(i + 1).zfill(2) for i in range(len(df))])
print("DATASET: first 5 rows\n", df.head())


# ---------- 1. MELT: wide -> long ----------
# Age and CTCAE grade are stacked into one 'Measure' column and one 'Value' column
long_df = df.melt(
    id_vars=['Patient', 'Country', 'Investigational Product Dose'],  # columns to keep as-is
    value_vars=['Age', 'CTCAE 6 Grade'],                             # columns to unpivot
    var_name='Measure',
    value_name='Value')
print("\n1. MELT: first 6 rows of the long format\n", long_df.head(6))


# ---------- 2. PIVOT: long -> wide ----------
# pivot() reverses the melt; each Patient + Measure pair is unique, so it works
wide_df = long_df.pivot(index='Patient', columns='Measure', values='Value')
print("\n2. PIVOT: back to wide format\n", wide_df.head())

# pivot_table() aggregates repeated pairs: how often each event occurred per country
ae_by_country = df.pivot_table(index='Country', columns='Adverse Event',
                               values='CTCAE 6 Grade', aggfunc='count', fill_value=0)
print("\n2. PIVOT_TABLE: adverse event counts by country\n", ae_by_country)


# ---------- 3. AGGREGATION ----------
# Summarize one column with several functions
print("\n3. AGGREGATION: mean, min and max grade\n",
      df['CTCAE 6 Grade'].agg(['mean', 'min', 'max']))

# Different functions for different columns (NaN = function not requested for that column)
print("\n3. AGGREGATION: different functions per column\n",
      df.agg({'Age': ['mean', 'median'], 'CTCAE 6 Grade': ['mean', 'max']}))


# ---------- 4. ITERATION ----------
# iterrows(): loop over rows one at a time
print("\n4. ITERATION: first 3 rows with iterrows()")
for index, row in df.head(3).iterrows():
    print(f"Row {index}: patient {row['Patient']} from {row['Country']} had {row['Adverse Event']}")

# zip(): loop over several columns together; flag severe events
print("\n4. ITERATION: severe events (grade 4 or higher)")
for patient, event, grade in zip(df['Patient'], df['Adverse Event'], df['CTCAE 6 Grade']):
    if grade >= 4:
        print(f"{patient} had {event} at grade {grade}")

# Iterating over groups: each loop gives a country name and its sub-table
print("\n4. ITERATION: summary for each country")
for country, group in df.groupby('Country'):
    print(f"{country}: {len(group)} events, mean grade {group['CTCAE 6 Grade'].mean():.2f}")


# ---------- 5. GROUPBY ----------
# Mean severity by dose
print("\n5. GROUPBY: mean grade by dose\n",
      df.groupby('Investigational Product Dose')['CTCAE 6 Grade'].mean())

# Several summaries per country at once (named aggregation)
print("\n5. GROUPBY: summary per country\n", df.groupby('Country').agg(
    events=('Patient', 'count'),
    avg_age=('Age', 'mean'),
    avg_grade=('CTCAE 6 Grade', 'mean'),
    max_grade=('CTCAE 6 Grade', 'max')))

# Group by two columns
print("\n5. GROUPBY: mean grade by dose and gender\n",
      df.groupby(['Investigational Product Dose', 'Gender'])['CTCAE 6 Grade'].mean())

DATASET: first 5 rows
   Patient Country Gender  Age Investigational Product Dose  \
0     P01     USA      M   62                     10 mg/kg   
1     P02     USA      F   55                      5 mg/kg   
2     P03     USA      M   71                     20 mg/kg   
3     P04     USA      F   48                     10 mg/kg   
4     P05   Italy      F   67                     20 mg/kg   

                   Adverse Event  CTCAE 6 Grade  
0                   Hypertension              2  
1                      Headaches              1  
2  Elevated liver function tests              3  
3                 Acneiform rash              2  
4                        Ascites              3  

1. MELT: first 6 rows of the long format
   Patient Country Investigational Product Dose Measure  Value
0     P01     USA                     10 mg/kg     Age     62
1     P02     USA                      5 mg/kg     Age     55
2     P03     USA                     20 mg/kg     Age     71
3     P04    